# 10 — Repaired evaluation, progress and detached launch

This notebook controls the authorized **8-hour AHR-only repair run**. Notebook **08** verifies immediate capture threats and repairs search; **09** mixes verified tactical policy targets with complete-game learning and selects checkpoints with held-out diagnostics; **11** lets you play the revised model. Notebooks 00–07 and the completed 12-hour run remain intact.

**Inputs:** tagged definitions from 00–03, 05, 08 and 09; their validation receipts; `config/ahr_v2_repair_training.json`; GPU 1. **Outputs:** full resumable state in `checkpoints/<run_id>/`; progress, reviews and the worker notebook in `reports/ahr_v2/`; separate revised active-run pointers. **Runtime:** normal Run All takes about a minute; training consumes 8 active hours plus checkpoint and development-review overhead.

GNU Go level 1 is used only for development evaluation, never training labels. Reviews use **128 visits per move**, without a smaller per-move clock limit. Games have a 900-second safety limit; timeout and ply cutoffs remain UNKNOWN. Win rate, captured stones, large captures and final surviving stones are reported separately. Repeated seeded game trajectories can be correlated. These are development measurements, not a final lockbox strength claim.

**Recovery:** resume the same run ID and exact configuration with `go2_launch_repair(PROJECT, resume=True)` after the old session ends. Clear a requested `checkpoints/<run_id>/STOP` marker first. Resume verifies source/config/payload hashes and restores optimizer, scheduler, scaler, all used RNGs, replay, tactical sampler, active game and pending search. Partial interrupted events never overwrite the last valid checkpoint. One writer owns each run; latest two valid checkpoints and the held-out best are retained.

The detached tmux session survives your connection. Workspace storage is the persistent boot disk; instance deletion policy is unverified. Off-instance backup remains deferred at the user's request until their storage destination is ready. No service is published.


## Load checkpoint contracts

In [ ]:
from pathlib import Path
import nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'AGENTS.md').is_file() and PROJECT!=PROJECT.parent:PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())


## Evaluation and launch definitions

In [ ]:
import shlex, subprocess, traceback
from zoneinfo import ZoneInfo

GO2_REPAIR_SEQUENCE=('00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','05_evaluation_and_progress.ipynb','08_tactical_constraints_and_search.ipynb','09_calibrated_selfplay_training.ipynb','10_repaired_evaluation_and_launch.ipynb','11_play_revised_player.ipynb')

def go2_repair_control_sources(project):
    project=Path(project)
    return {GO2_ROOT+'/'+name:go2_code_sha(project/GO2_ROOT/name) for name in GO2_REPAIR_SEQUENCE}

def go2_repair_evaluation_sources(project):
    result=go2_source_identity(project)
    for name in ('05_evaluation_and_progress.ipynb','08_tactical_constraints_and_search.ipynb','09_calibrated_selfplay_training.ipynb','10_repaired_evaluation_and_launch.ipynb'):
        result[GO2_ROOT+'/'+name]=go2_code_sha(Path(project)/GO2_ROOT/name)
    return result

def go2_repair_namespace(project):
    namespace={'__name__':'ahr_v2_repair_exports'}
    for name in GO2_REPAIR_SEQUENCE[:7]:go2_import_exports(Path(project)/GO2_ROOT/name,namespace)
    namespace['go2_eval_sources']=go2_repair_evaluation_sources
    namespace['go2_load_player']=namespace['go2_load_repair_player']
    return namespace

def go2_repair_evaluation_config(games=4,seed=2026100710):
    return {'games':int(games),'board_size':9,'komi':7.5,'simulations':128,'batch_size':8,'seed':int(seed),
            'max_plies':600,'game_time_seconds':900,'move_time_seconds':None,'purpose':'development'}

def go2_repair_capture_diagnostics(namespace,games):
    items=[]
    for game in games:
        state=namespace['go2_initial_state']();removed=0;events=[]
        for ply,move in enumerate(game['trace'],1):
            before=sum(value==game['ahr_color'] for value in state.board)
            successor=namespace['go2_step'](state,int(move['action']))
            after=sum(value==game['ahr_color'] for value in successor.board)
            if state.to_play!=game['ahr_color'] and after<before:
                count=before-after;removed+=count;events.append({'ply':ply,'stones':count})
            state=successor
        items.append({'game_index':game['game_index'],'ahr_color':game['ahr_color'],'outcome':game['outcome'],
            'ahr_stones_captured':removed,'large_capture_events':[event for event in events if event['stones']>=4],
            'ahr_stones_remaining':sum(value==game['ahr_color'] for value in state.board),
            'trajectory_sha256':go2_digest([move['action'] for move in game['trace']])})
    return {'games':items,'total_ahr_stones_captured':sum(item['ahr_stones_captured'] for item in items),
        'wiped_out_games':sum(item['ahr_stones_remaining']==0 and item['ahr_stones_captured']>0 and item['outcome']!='UNKNOWN' for item in items),
        'unique_game_trajectories':len({item['trajectory_sha256'] for item in items}),
        'interpretation':'Capture counts describe observed play, not a proof of optimal moves. Seeded trajectories can be correlated; Wilson intervals are descriptive development evidence.'}

def go2_repair_evaluate(project=None,run_id=None,eval_id=None,config=None,which='latest',device='cpu',namespace=None):
    project=go2_project(project);namespace=namespace or go2_repair_namespace(project)
    report=namespace['go2_evaluate'](project,run_id,eval_id,config or go2_repair_evaluation_config(),which,device)
    if report['config']['board_size']==9:
        report['tactical_diagnostics']=go2_repair_capture_diagnostics(namespace,report['games'])
    report['repaired_search_sources']=go2_repair_evaluation_sources(project)
    report['comparison_note']='The prior run used 32 visits; this revised player uses 128. These are player-configuration comparisons, not an equal-compute ablation.'
    go2_atomic_json(project/'reports/ahr_v2/evaluations'/report['eval_id']/'report.json',report)
    return report

def go2_repair_preflight(project,config):
    project=Path(project);sources=go2_repair_control_sources(project);checks={}
    for name,path in [('tactical','tactical_search_validation.json'),('training','repair_training_validation.json'),('play','repair_play_validation.json'),('arena','repair_evaluation_validation.json')]:
        receipt=json.loads((project/'reports/ahr_v2'/path).read_text())
        checks[name+'_validation']=receipt.get('status')=='PASS' and bool(receipt.get('sources')) and all(sources.get(path)==digest for path,digest in receipt['sources'].items())
        if name=='training':
            checks['cuda_exact_resume']=any(item.get('device')=='cuda:1' and item.get('fresh_kernel_exact_scientific_state') is True for item in receipt.get('fixtures',[]))
            proof=receipt.get('production_shape',{})
            checks['real_9x9_training']=proof.get('status')=='PASS' and proof.get('completed_games',0)>0 and proof.get('optimizer_steps',0)>0
    checks['gpu_1']=config['device']=='cuda:1' and torch.cuda.is_available() and torch.cuda.device_count()>=2
    checks['tmux']=subprocess.run(['tmux','-V'],capture_output=True,text=True).returncode==0
    checks['space']=os.statvfs(project).f_bavail*os.statvfs(project).f_frsize>20*1024**3
    checks['eight_hour_budget']=config['time_budget_hours']==8
    namespace=go2_repair_namespace(project)
    namespace['go2_validate_repair_training_config'](config)
    namespace['go2_opponent_pin'](namespace['Go2EvalConfig'](**go2_repair_evaluation_config()))
    if not all(checks.values()):raise RuntimeError('Repaired-run preflight failed: '+repr(checks))
    return {'status':'PASS','checks':checks,'sources':sources,'config_sha256':go2_digest(config),'checked_at_utc':go2_now()}

def go2_repair_review(project,config,namespace,control,final=False):
    project=Path(project);run_id=config['run_id'];snapshot,_=go2_load_checkpoint(project/'checkpoints'/run_id)
    if snapshot is None or snapshot['metrics']['optimizer_step']<=0:return control
    step=snapshot['metrics']['optimizer_step'];eval_id=run_id+('-final-dev-' if final else '-periodic-dev-')+str(step)
    control['active_review']={'eval_id':eval_id,'started_at_utc':go2_now(),'status':'RUNNING'}
    path=project/'reports/ahr_v2'/run_id/'control.json';go2_atomic_json(path,control)
    review={'eval_id':eval_id,'optimizer_step':step,'status':'PASS','final_development_review':bool(final),
        'report_path':'reports/ahr_v2/evaluations/'+eval_id+'/report.json'}
    rng=go2_capture_rng(cuda_devices=(1,))
    try:
        report=go2_repair_evaluate(project,run_id,eval_id,go2_repair_evaluation_config(8 if final else 2),device='cpu',namespace=namespace)
        review['summary']=report['summary'];review['tactical_diagnostics']={k:v for k,v in report.get('tactical_diagnostics',{}).items() if k!='games'}
    except Exception as error:review.update(status='ERROR',error=str(error),traceback=traceback.format_exc())
    finally:go2_restore_rng(rng)
    review['finished_at_utc']=go2_now();control['active_review']=None
    control.setdefault('reviews',[]).append(review);control['last_review_at_utc']=review['finished_at_utc']
    go2_atomic_json(path,control);return control

def go2_repair_pipeline(project,config):
    project=Path(project);namespace=go2_repair_namespace(project);path=project/'reports/ahr_v2'/config['run_id']/'control.json'
    control=json.loads(path.read_text()) if path.is_file() else {'run_id':config['run_id'],'reviews':[],'last_review_at_utc':go2_now()}
    go2_atomic_json(path,control)
    def on_checkpoint(state,entry):
        nonlocal control
        due=(dt.datetime.now(dt.timezone.utc)-dt.datetime.fromisoformat(control['last_review_at_utc'])).total_seconds()>=7200
        if due and state['metrics']['optimizer_step']>0 and not (project/'checkpoints'/config['run_id']/'STOP').exists():
            control=go2_repair_review(project,config,namespace,control)
    result=namespace['go2_run_repair_training'](project,config,on_checkpoint=on_checkpoint)
    if not (project/'checkpoints'/config['run_id']/'STOP').exists() and not any(item.get('final_development_review') for item in control['reviews']):
        control=go2_repair_review(project,config,namespace,control,True)
    return result

def go2_repair_worker(project,config_path,launch_path):
    project=Path(project);launch=json.loads(Path(launch_path).read_text());config=json.loads(Path(config_path).read_text())
    if go2_repair_control_sources(project)!=launch['sources'] or go2_digest(config)!=launch['config_sha256']:
        raise RuntimeError('Pinned source/config changed; refusing to launch or resume.')
    path=project/'reports/ahr_v2'/config['run_id']/'worker.json'
    worker={'status':'RUNNING','pid':os.getpid(),'run_id':config['run_id'],'started_at_utc':go2_now(),'session':launch['session']}
    go2_atomic_json(path,worker)
    try:result=go2_repair_pipeline(project,config)
    except BaseException as error:
        go2_atomic_json(path,{**worker,'status':'ERROR','finished_at_utc':go2_now(),'error':str(error),'traceback':traceback.format_exc()});raise
    go2_atomic_json(path,{**worker,'status':'RETURNED','finished_at_utc':go2_now()});return result

def go2_launch_repair(project,config_path='config/ahr_v2_repair_training.json',session='ahr-v2-repair-8h',resume=False):
    project=Path(project).resolve();config_path=project/config_path;config=json.loads(config_path.read_text());preflight=go2_repair_preflight(project,config)
    if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0:raise RuntimeError('Session already exists; inspect or attach instead of starting a second writer.')
    directory=project/'checkpoints'/config['run_id']
    if directory.exists():
        saved,_=go2_load_checkpoint(directory)
        if saved is not None and not resume:raise RuntimeError('Checkpoint exists; use resume=True with the exact config.')
    report_dir=project/'reports/ahr_v2'/config['run_id'];report_dir.mkdir(parents=True,exist_ok=True)
    runner_path=report_dir/'detached_worker.ipynb';launch_path=report_dir/'launch.json';log_path=report_dir/'tmux.log'
    launch={**preflight,'run_id':config['run_id'],'session':session,'launched_at_utc':go2_now(),'requested_hours':8,
        'resume':bool(resume),'config_path':str(config_path.relative_to(project)),'worker_notebook':str(runner_path.relative_to(project)),
        'log_path':str(log_path.relative_to(project)),'checkpoint_path':'checkpoints/'+config['run_id'],
        'play_notebook':GO2_ROOT+'/11_play_revised_player.ipynb','attach_command':'tmux attach -t '+session,'detach_keys':'Ctrl-b, then d'}
    lines=['from pathlib import Path','import nbformat','PROJECT=Path('+repr(str(project))+')',
        'for name in ["00_contracts_and_recovery.ipynb","10_repaired_evaluation_and_launch.ipynb"]:',
        '    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:',
        '        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]):exec(compile(cell.source,name+":ahr-export","exec"),globals())',
        'state=go2_repair_worker(PROJECT,Path('+repr(str(config_path))+'),Path('+repr(str(launch_path))+'))',
        'print("Repaired worker returned. Inspect status.json and worker.json.")']
    book=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}},cells=[nbformat.v4.new_markdown_cell('# Detached repaired AHR v2 worker — generated by notebook 10.'),nbformat.v4.new_code_cell(chr(10).join(lines))])
    go2_atomic_bytes(runner_path,nbformat.writes(book).encode());go2_atomic_json(launch_path,launch)
    args=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(runner_path)]
    command=' '.join(shlex.quote(arg) for arg in args)+' >> '+shlex.quote(str(log_path))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True,capture_output=True,text=True)
    go2_atomic_json(project/'reports/ahr_v2/repair_active_run.json',launch);go2_atomic_json(project/'config/ahr_v2_repair_active_run.json',launch)
    return launch


## Validate revised arena recovery

In [ ]:
def go2_repair_arena_validation(project=None):
    project=go2_project(project);namespace=go2_repair_namespace(project);torch.set_num_threads(2)
    token=uuid.uuid4().hex[:8];full_id='ahr-v2-repair-arena-fixture-'+token+'-full';resume_id='ahr-v2-repair-arena-fixture-'+token+'-resume'
    player=namespace['_go2_eval_fixture_player'](project)
    config=namespace['Go2EvalConfig'](games=2,board_size=3,simulations=4,batch_size=2,max_plies=8,game_time_seconds=None,purpose='validation_fixture')
    kwargs={'config':config,'_player':player,'_opponent_factory':namespace['_Go2PassOpponent'],'_search_evaluator':namespace['_go2_pass_evaluator'](player[0])}
    full=namespace['go2_evaluate'](project,eval_id=full_id,**kwargs)
    partial=namespace['go2_evaluate'](project,eval_id=resume_id,_max_events=2,**kwargs)
    assert partial['status']=='PAUSED'
    saved,_=go2_load_checkpoint(project/'checkpoints'/resume_id);assert saved['search'] is not None
    fixture_path=project/'reports/ahr_v2/fixtures'/('repair-arena-resume-'+token+'.ipynb')
    lines=['from pathlib import Path','import nbformat','PROJECT=Path('+repr(str(project))+')',
        'for name in ["00_contracts_and_recovery.ipynb","10_repaired_evaluation_and_launch.ipynb"]:',
        '    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:',
        '        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]):exec(cell.source,globals())',
        'namespace=go2_repair_namespace(PROJECT)','torch.set_num_threads(2)','player=namespace["_go2_eval_fixture_player"](PROJECT)',
        'report=namespace["go2_evaluate"](PROJECT,eval_id='+repr(resume_id)+',_opponent_factory=namespace["_Go2PassOpponent"],_search_evaluator=namespace["_go2_pass_evaluator"](player[0]))',
        'assert report["status"]=="COMPLETE"']
    book=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}},cells=[nbformat.v4.new_code_cell(chr(10).join(lines))])
    go2_atomic_bytes(fixture_path,nbformat.writes(book).encode())
    subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=120',str(fixture_path)],check=True,capture_output=True,text=True)
    after=json.loads((project/'reports/ahr_v2/evaluations'/resume_id/'report.json').read_text())
    assert full['games']==after['games'] and full['summary']==after['summary']
    inputs=[{'outcome':label,'ahr_color':1} for label in ('WIN','LOSS','DRAW','UNKNOWN')]
    summary=namespace['go2_eval_summary'](inputs);assert summary['unknown']==1 and summary['complete_games']==3
    gnu_config=namespace['Go2EvalConfig'](games=2,simulations=4,batch_size=2,max_plies=4,game_time_seconds=60,purpose='validation_fixture')
    smoke=namespace['go2_evaluate'](project,eval_id='ahr-v2-repair-gnu-smoke-'+token,config=gnu_config,_player=player)
    assert smoke['summary']['unknown']==2 and smoke['summary']['complete_games']==0
    receipt={'status':'PASS','sources':go2_repair_evaluation_sources(project),'checks':{'repaired_search_game_resume_fresh_kernel':True,'UNKNOWN_not_loss_or_draw':True,'real_GNU_Go_revised_search_smoke':True,'fixed_128_visit_review_configuration':go2_repair_evaluation_config()['simulations']==128},'resume_notebook':str(fixture_path.relative_to(project)),'opponent':namespace['go2_opponent_pin'](gnu_config),'validated_at_utc':go2_now(),'scope':'Disposable correctness fixtures, not player strength.'}
    go2_atomic_json(project/'reports/ahr_v2/repair_evaluation_validation.json',receipt);return receipt


In [ ]:
receipt=go2_repair_arena_validation(PROJECT)
from IPython.display import HTML,display
rows=''.join('<tr><td>'+name+'</td><td>PASS</td></tr>' for name in receipt['checks'])
display(HTML('<h3>Revised arena checks passed</h3><table>'+rows+'</table>'))


## Inspect the revised run

In [ ]:
config_path=PROJECT/'config/ahr_v2_repair_training.json'
active_path=PROJECT/'reports/ahr_v2/repair_active_run.json'
namespace=go2_repair_namespace(PROJECT)
if config_path.is_file():
    TRAINING_CONFIG=json.loads(config_path.read_text())
    preflight=go2_repair_preflight(PROJECT,TRAINING_CONFIG)
    display(HTML('<h3>Repair preflight: '+preflight['status']+'</h3><p>8 active hours · GPU 1 · pure AHR learning · detached tmux</p>'))
    progress=namespace['go2_progress'](PROJECT,TRAINING_CONFIG['run_id'])
    namespace['go2_progress_display'](progress)
    print('Verified tactical training rows seen:',progress.get('tactical_rows_seen',0))
    calibration=progress.get('last_calibration')
    if calibration:
        rows=[]
        for split in ('train','dev'):
            values=calibration['splits'][split]
            if values.get('rows',0):
                rows.append('<tr><td>'+split+'</td><td>'+str(values['games'])+'</td><td>'+str(values['rows'])+'</td><td>'+format(values['accuracy'],'.1%')+'</td><td>'+format(values['brier'],'.3f')+'</td><td>'+format(values['score_mae']*88.5,'.1f')+'</td></tr>')
        display(HTML('<h3>Current held-out calibration sample</h3><table><tr><th>Split</th><th>Games</th><th>Positions</th><th>Outcome accuracy</th><th>Brier error</th><th>Score error (points)</th></tr>'+''.join(rows)+'</table><p>Outcomes describe the played self-play continuations. These measurements do not prove optimal position values or GNU Go strength.</p>'))
    else:
        print('Held-out calibration will appear once reserved development games are available. Best-player strength has not been established.')
    reviews=progress.get('development_reviews',[])
    if reviews:
        rows=[]
        for review in reviews:
            summary=review.get('summary',{})
            rows.append('<tr><td>'+str(review['optimizer_step'])+'</td><td>'+str(summary.get('wins','—'))+'</td><td>'+str(summary.get('losses','—'))+'</td><td>'+str(summary.get('draws','—'))+'</td><td>'+str(summary.get('unknown','—'))+'</td></tr>')
        display(HTML('<h3>GNU Go level 1 development reviews</h3><table><tr><th>Optimizer step</th><th>Wins</th><th>Losses</th><th>Draws</th><th>UNKNOWN</th></tr>'+''.join(rows)+'</table><p>Each row belongs to a different pinned checkpoint; games are not pooled into a current-player win rate.</p>'))
else:
    TRAINING_CONFIG=None
    print('Repair configuration will be frozen after the training and play checks pass.')
if active_path.is_file():
    active=json.loads(active_path.read_text())
    display({key:active[key] for key in ['run_id','session','requested_hours','checkpoint_path','attach_command']})
    print('Launched:',dt.datetime.fromisoformat(active['launched_at_utc']).astimezone(ZoneInfo('Europe/Dublin')).strftime('%Y-%m-%d %H:%M %Z'))


## Start, stop, resume and play

Run All validates the revised arena and shows readiness. The authorized long run is launched once with `go2_launch_repair(PROJECT)` after all receipts pass; Run All does not launch another writer.

Attach with `tmux attach -t ahr-v2-repair-8h`. Detach with **Ctrl-b**, then **d**. To stop at a safe boundary, create `checkpoints/<run_id>/STOP`. To resume, clear that marker and use `go2_launch_repair(PROJECT, resume=True)` with the same configuration after the old session ends.

Open **11 — Play revised player**, Run All, choose a colour and click **New game**. Its opponent is a frozen snapshot, so training never changes the model in the middle of your game. Notebook **06** remains available for the old player.

The tactical fixtures establish bounded capture/defence behaviour. Whether the revised system wins at least half its games against GNU Go is an outcome to measure.


In [ ]:
print('Control notebook ready. The detached launch is an explicit one-time action.')